# Prática — Árvore AVL

**Estruturas de Dados Básicas II** · Atividade assíncrona.

## Instruções

1. Preencha as células marcadas com `# SEU CÓDIGO AQUI` (apague o `raise NotImplementedError`).
2. Execute as células de **teste** logo abaixo de cada questão. Todas devem terminar com ✅.
3. Responda as perguntas discursivas nas células **✍️ Sua resposta**.
4. **Não use** bibliotecas prontas de árvore, nem `sorted`, `dict` ou `set` para *resolver* as questões (usar `collections.deque` como fila é permitido).
5. **Antes de enviar:** use *Kernel → Restart & Run All* e confira se todas as saídas estão visíveis. Envie o arquivo `.ipynb`.

> Nome: Richard Matheus Bezerra Ataliba · Matrícula: 20240023956

---
## Ponto de partida — reaproveite a sua Árvore Binária de Busca

Esta prática **reaproveita a Árvore Binária de Busca que você implementou na atividade anterior**. Cole na célula abaixo as classes `No` e `ArvoreBinariaBusca` exatamente como entregou (com as correções, se houver).

A célula de verificação logo depois confere o básico da sua ABB. Se ela falhar, corrija a ABB **antes** de começar a AVL: a AVL vai herdar desses métodos.

In [1]:
# ===== COLE AQUI as classes No e ArvoreBinariaBusca da Prática ABB =====
# (copiadas da Prática ABB, sem alterações)
class No:
    def __init__(self, chave, valor=None):
        self.chave = chave
        self.valor = valor
        self.esquerda = None
        self.direita = None

    def __repr__(self):
        return f"No({self.chave!r})"


from collections import deque


class ArvoreBinariaBusca:
    def __init__(self):
        self.raiz = None
        self.tamanho = 0
        self.comparacoes = 0   # nós visitados na última chamada a buscar()

    def __len__(self):
        return self.tamanho

    # ---------- Parte A: operações básicas ----------

    def adicionar(self, chave, valor=None):
        if self.raiz is None:
            self.raiz = No(chave, valor)
            self.tamanho = 1
            return
        atual = self.raiz
        while True:
            if chave == atual.chave:
                atual.valor = valor
                return
            if chave < atual.chave:
                if atual.esquerda is None:
                    atual.esquerda = No(chave, valor)
                    self.tamanho += 1
                    return
                atual = atual.esquerda
            else:
                if atual.direita is None:
                    atual.direita = No(chave, valor)
                    self.tamanho += 1
                    return
                atual = atual.direita

    def buscar(self, chave):
        self.comparacoes = 0
        atual = self.raiz
        while atual is not None:
            self.comparacoes += 1
            if chave == atual.chave:
                return atual.valor
            if chave < atual.chave:
                atual = atual.esquerda
            else:
                atual = atual.direita
        return None

    def contem(self, chave):
        atual = self.raiz
        while atual is not None:
            if chave == atual.chave:
                return True
            if chave < atual.chave:
                atual = atual.esquerda
            else:
                atual = atual.direita
        return False

    def remover(self, chave):
        pai = None
        atual = self.raiz
        eh_esquerda = False
        while atual is not None and atual.chave != chave:
            pai = atual
            if chave < atual.chave:
                atual = atual.esquerda
                eh_esquerda = True
            else:
                atual = atual.direita
                eh_esquerda = False
        if atual is None:
            return False

        # Caso 2 filhos: substitui pelo sucessor e remove o sucessor
        if atual.esquerda is not None and atual.direita is not None:
            suc_pai = atual
            sucessor = atual.direita
            while sucessor.esquerda is not None:
                suc_pai = sucessor
                sucessor = sucessor.esquerda
            atual.chave = sucessor.chave
            atual.valor = sucessor.valor
            # agora remove o sucessor (tem no máximo filho direito)
            pai = suc_pai
            atual = sucessor
            eh_esquerda = (pai.esquerda is atual)

        # Caso 0 ou 1 filho
        filho = atual.esquerda if atual.esquerda is not None else atual.direita
        if pai is None:
            self.raiz = filho
        elif eh_esquerda:
            pai.esquerda = filho
        else:
            pai.direita = filho
        self.tamanho -= 1
        return True

    def minimo(self):
        if self.raiz is None:
            return None
        atual = self.raiz
        while atual.esquerda is not None:
            atual = atual.esquerda
        return atual.chave

    def maximo(self):
        if self.raiz is None:
            return None
        atual = self.raiz
        while atual.direita is not None:
            atual = atual.direita
        return atual.chave

    def profundidade(self, chave):
        profund = 0
        atual = self.raiz
        while atual is not None:
            if chave == atual.chave:
                return profund
            if chave < atual.chave:
                atual = atual.esquerda
            else:
                atual = atual.direita
            profund += 1
        return None

    # ---------- Parte B: percursos ----------

    def em_ordem(self, processar):
        def _em_ordem(no):
            if no is None:
                return
            _em_ordem(no.esquerda)
            processar(no)
            _em_ordem(no.direita)
        _em_ordem(self.raiz)

    def pre_ordem(self, processar):
        def _pre_ordem(no):
            if no is None:
                return
            processar(no)
            _pre_ordem(no.esquerda)
            _pre_ordem(no.direita)
        _pre_ordem(self.raiz)

    def pos_ordem(self, processar):
        def _pos_ordem(no):
            if no is None:
                return
            _pos_ordem(no.esquerda)
            _pos_ordem(no.direita)
            processar(no)
        _pos_ordem(self.raiz)

    def em_largura(self, processar):
        if self.raiz is None:
            return
        fila = deque([self.raiz])
        while fila:
            no = fila.popleft()
            processar(no)
            if no.esquerda is not None:
                fila.append(no.esquerda)
            if no.direita is not None:
                fila.append(no.direita)

In [2]:
# ===== FERRAMENTAS AUXILIARES (não altere) =====

def coletar(percurso):
    """Transforma qualquer percurso em uma lista de chaves. Ex.: coletar(arvore.em_ordem)"""
    chaves = []
    percurso(lambda no: chaves.append(no.chave))
    return chaves


def mostrar(arvore, max_nos=60):
    """Desenha a árvore "deitada" (a raiz fica à esquerda e a subárvore direita fica em cima).
    Ao lado de cada chave aparecem a altura guardada no nó e o FB."""
    if arvore.raiz is None:
        print("(árvore vazia)")
        return
    if len(arvore) > max_nos:
        print(f"(árvore com {len(arvore)} nós: grande demais para desenhar)")
        return

    def h(no):
        return -1 if no is None else getattr(no, "altura", "?")

    def desenhar(no, nivel):
        if no is None:
            return
        desenhar(no.direita, nivel + 1)
        try:
            fb = h(no.esquerda) - h(no.direita)
        except TypeError:
            fb = "?"
        print("        " * nivel + f"{no.chave}  [h={h(no)}, FB={fb}]")
        desenhar(no.esquerda, nivel + 1)

    desenhar(arvore.raiz, 0)


def verificar_avl(arvore):
    """Confere, sem recursão, se a árvore é uma AVL válida:
    ordem de ABB, alturas guardadas corretas, |FB| <= 1 em todos os nós e tamanho certo.
    Levanta AssertionError explicando o primeiro problema encontrado."""
    # 1) percurso em-ordem iterativo: ordem de ABB, tipo dos nós e contagem
    chaves, pilha, atual = [], [], arvore.raiz
    while pilha or atual is not None:
        while atual is not None:
            assert isinstance(atual, AVLNode), (
                f"{atual!r} não é AVLNode: algum método está criando No em vez de AVLNode?")
            pilha.append(atual)
            atual = atual.esquerda
            assert len(pilha) <= len(arvore) + 1, "caminho maior que o número de nós: há um ciclo?"
        atual = pilha.pop()
        assert not chaves or chaves[-1] < atual.chave, (
            f"ordem de ABB violada perto da chave {atual.chave!r}")
        chaves.append(atual.chave)
        atual = atual.direita
    assert len(chaves) == len(arvore), (
        f"a árvore tem {len(chaves)} nós, mas len() diz {len(arvore)}")

    # 2) pós-ordem iterativo: recalcula as alturas e confere com o campo `altura`
    if arvore.raiz is None:
        return True
    altura_real = {}
    pilha = [(arvore.raiz, False)]
    while pilha:
        no, filhos_prontos = pilha.pop()
        if not filhos_prontos:
            pilha.append((no, True))
            for filho in (no.direita, no.esquerda):
                if filho is not None:
                    pilha.append((filho, False))
            continue
        he = -1 if no.esquerda is None else altura_real[id(no.esquerda)]
        hd = -1 if no.direita is None else altura_real[id(no.direita)]
        altura_real[id(no)] = 1 + max(he, hd)
        assert no.altura == altura_real[id(no)], (
            f"nó {no.chave!r}: altura guardada = {no.altura}, altura real = {altura_real[id(no)]}")
        assert abs(he - hd) <= 1, f"nó {no.chave!r} está desbalanceado (FB = {he - hd})"
    return True

In [3]:
# ===== VERIFICAÇÃO RÁPIDA DA SUA ABB (não altere) =====
def verificar_abb():
    assert "No" in globals() and "ArvoreBinariaBusca" in globals(), (
        "cole as classes No e ArvoreBinariaBusca na célula acima e execute-a")
    a = ArvoreBinariaBusca()
    for c in [50, 30, 70, 20, 40, 60, 80]:
        a.adicionar(c, f"v{c}")
    assert len(a) == 7 and a.raiz.chave == 50, "adicionar() da ABB com problema"
    assert a.buscar(60) == "v60" and a.comparacoes == 3, "buscar()/comparacoes da ABB com problema"
    assert a.minimo() == 20 and a.maximo() == 80 and a.profundidade(40) == 2
    assert coletar(a.em_ordem) == [20, 30, 40, 50, 60, 70, 80], "em_ordem() da ABB com problema"
    assert coletar(a.em_largura) == [50, 30, 70, 20, 40, 60, 80], "em_largura() da ABB com problema"
    print("✅ Sua ABB está pronta para ser herdada!")

verificar_abb()

✅ Sua ABB está pronta para ser herdada!


---
## Questão 1 — Implementando a AVL

Implemente a Árvore AVL usando as convenções da aula:

- **Altura:** folha tem altura **0**; árvore vazia (`None`) tem altura **−1**.
- **Fator de balanceamento:** `FB(n) = h(esquerda) − h(direita)`. A árvore é AVL se `FB ∈ {−1, 0, +1}` em **todos** os nós.
- **Tabela de decisão** (no nó com FB = ±2):

| FB do nó | FB do filho do lado mais alto | Rotação |
|---|---|---|
| +2 | ≥ 0 (esquerdo) | simples à direita |
| +2 | −1 (esquerdo) | dupla: esquerda no filho, direita no nó |
| −2 | ≤ 0 (direito) | simples à esquerda |
| −2 | +1 (direito) | dupla: direita no filho, esquerda no nó |

### Métodos

| Classe / método | Comportamento esperado |
|---|---|
| `AVLNode(chave, valor=None)` | Nó da ABB com um campo a mais: `altura` (um nó recém-criado é folha, altura 0). |
| `altura()` | Altura da árvore (−1 se vazia), em **O(1)**. |
| `atualizar_altura(no)` | Recalcula `no.altura` a partir das alturas dos filhos. |
| `fator_balanceamento(no)` | Retorna o FB do nó. |
| `rotacionar_direita(y)` / `rotacionar_esquerda(x)` | Fazem a rotação simples, atualizam as alturas (na ordem certa!), somam 1 em `self.rotacoes` e **retornam a nova raiz da subárvore**. |
| `balancear(no)` | Atualiza a altura do nó, aplica a rotação da tabela se o FB for ±2 e **retorna a raiz da subárvore** (o próprio nó, se nada mudou). Uma rotação dupla soma 2 em `self.rotacoes`. |
| `adicionar(chave, valor=None)` | Insere como na ABB e rebalanceia o caminho de volta até a raiz. Chave repetida só atualiza o valor. |
| `remover(chave)` | Remove como na ABB (dois filhos → **sucessor**) e rebalanceia o caminho de volta até a raiz. Retorna `True`/`False`. |

Mantenha `self.tamanho` atualizado.

> 💡 **Dica — herança.** Você não precisa reescrever a ABB. `AVLNode` pode **herdar de `No`** e só acrescentar o campo `altura`; `AVLTree` pode **herdar de `ArvoreBinariaBusca`** e sobrescrever apenas o que muda. Use `super().__init__(...)` nos construtores.
>
> Pense em quais métodos da ABB **só leem** a árvore (`buscar`, `contem`, `minimo`, `maximo`, `profundidade`, os quatro percursos…): eles funcionam na AVL sem nenhuma alteração. Já `adicionar` e `remover` **precisam** ser sobrescritos. Por quê? (Duas razões: o que eles criam e o que eles não fazem.)

> 💡 **Dica — recursão.** Na ABB, `adicionar` precisava ser iterativo por causa da árvore em fila. Na AVL a altura é O(log n), então uma versão **recursiva que devolve a nova raiz da subárvore** é segura e deixa o rebalanceamento natural: ele acontece no retorno da recursão, de baixo para cima.

> 💡 Use `mostrar(arvore)` para desenhar árvores pequenas enquanto depura, e `verificar_avl(arvore)` para conferir se a sua árvore continua AVL.

In [4]:
class AVLNode(No):
    def __init__(self, chave, valor=None):
        # Reaproveita chave, valor, esquerda e direita do nó da ABB
        super().__init__(chave, valor)
        # Nó recém-criado é folha: altura 0 (None tem altura -1)
        self.altura = 0


class AVLTree(ArvoreBinariaBusca):
    def __init__(self):
        super().__init__()
        self.rotacoes = 0   # rotações simples feitas desde a criação (uma dupla conta 2)

    @staticmethod
    def _altura(no):
        """Altura de um nó, com a convenção da aula: None -> -1."""
        return -1 if no is None else no.altura

    def altura(self):
        # O(1): a altura da árvore é a altura guardada na raiz
        return self._altura(self.raiz)

    def atualizar_altura(self, no):
        # Altura = 1 + maior altura entre os filhos (as dos filhos já devem estar corretas)
        no.altura = 1 + max(self._altura(no.esquerda), self._altura(no.direita))

    def fator_balanceamento(self, no):
        # FB = h(esquerda) - h(direita)
        return self._altura(no.esquerda) - self._altura(no.direita)

    def rotacionar_direita(self, y):
        #       y              x
        #      / \            / \
        #     x   T3   ->   T1   y
        #    / \                / \
        #  T1   T2            T2   T3
        x = y.esquerda
        t2 = x.direita
        x.direita = y
        y.esquerda = t2          # T2 troca de pai
        # Primeiro o nó que desceu (y), depois a nova raiz (x), que depende de y
        self.atualizar_altura(y)
        self.atualizar_altura(x)
        self.rotacoes += 1
        return x

    def rotacionar_esquerda(self, x):
        #     x                  y
        #    / \                / \
        #  T1   y      ->      x   T3
        #      / \            / \
        #    T2   T3        T1   T2
        y = x.direita
        t2 = y.esquerda
        y.esquerda = x
        x.direita = t2           # T2 troca de pai
        self.atualizar_altura(x)
        self.atualizar_altura(y)
        self.rotacoes += 1
        return y

    def balancear(self, no):
        self.atualizar_altura(no)
        fb = self.fator_balanceamento(no)
        if fb == 2:                                        # pesado à esquerda
            if self.fator_balanceamento(no.esquerda) < 0:  # caso LR: dupla
                no.esquerda = self.rotacionar_esquerda(no.esquerda)
            return self.rotacionar_direita(no)             # caso LL (ou 2ª metade do LR)
        if fb == -2:                                       # pesado à direita
            if self.fator_balanceamento(no.direita) > 0:   # caso RL: dupla
                no.direita = self.rotacionar_direita(no.direita)
            return self.rotacionar_esquerda(no)            # caso RR (ou 2ª metade do RL)
        return no                                          # já balanceado

    def adicionar(self, chave, valor=None):
        # Recursão que devolve a nova raiz de cada subárvore:
        # o rebalanceamento acontece na volta, de baixo para cima.
        def _adicionar(no):
            if no is None:
                self.tamanho += 1
                return AVLNode(chave, valor)
            if chave == no.chave:
                no.valor = valor           # chave repetida: só atualiza o valor
                return no                  # (alturas não mudam, nada a rebalancear)
            if chave < no.chave:
                no.esquerda = _adicionar(no.esquerda)
            else:
                no.direita = _adicionar(no.direita)
            return self.balancear(no)

        self.raiz = _adicionar(self.raiz)

    def remover(self, chave):
        removido = False

        def _remover(no, chave):
            nonlocal removido
            if no is None:
                return None                # chave não encontrada nesta subárvore
            if chave < no.chave:
                no.esquerda = _remover(no.esquerda, chave)
            elif chave > no.chave:
                no.direita = _remover(no.direita, chave)
            else:
                # Achou o nó a remover
                if no.esquerda is None or no.direita is None:
                    # 0 ou 1 filho: o filho (ou None) sobe no lugar do nó
                    removido = True
                    return no.esquerda if no.esquerda is not None else no.direita
                # 2 filhos: copia o sucessor (menor da subárvore direita)
                # e remove o sucessor da subárvore direita
                sucessor = no.direita
                while sucessor.esquerda is not None:
                    sucessor = sucessor.esquerda
                no.chave, no.valor = sucessor.chave, sucessor.valor
                no.direita = _remover(no.direita, sucessor.chave)
            return self.balancear(no)

        self.raiz = _remover(self.raiz, chave)
        if removido:
            self.tamanho -= 1
        return removido

In [5]:
# ===== TESTES DA QUESTÃO 1 — nó, altura, FB e rotações (não altere) =====

def _no(t, chave, esq=None, dir=None):
    """Monta um nó à mão (para os testes) e acerta a altura dele."""
    n = AVLNode(chave)
    n.esquerda, n.direita = esq, dir
    t.atualizar_altura(n)
    return n

def testar_q1_rotacoes():
    t = AVLTree()
    assert t.altura() == -1 and len(t) == 0 and t.rotacoes == 0, "árvore nova deve estar vazia"
    n = AVLNode(5, "x")
    assert isinstance(n, No), "AVLNode deve herdar de No"
    assert (n.chave, n.valor, n.altura, n.esquerda, n.direita) == (5, "x", 0, None, None)

    # 30 com filho esquerdo 20, que tem filho esquerdo 10 (caso LL)
    y = _no(t, 30, _no(t, 20, _no(t, 10)))
    assert y.altura == 2 and t.fator_balanceamento(y) == 2, "altura/FB errados"
    assert t.fator_balanceamento(y.esquerda) == 1 and t.fator_balanceamento(y.esquerda.esquerda) == 0

    r = t.rotacionar_direita(y)
    assert r.chave == 20 and r.esquerda.chave == 10 and r.direita.chave == 30, "rotação à direita errada"
    assert (r.altura, r.esquerda.altura, r.direita.altura) == (1, 0, 0), "alturas erradas após a rotação"
    assert t.rotacoes == 1, "rotacionar_direita deve somar 1 em self.rotacoes"

    # T2 troca de pai: 30( 20(10, 25), 40 ) -> 20( 10, 30(25, 40) )
    y = _no(t, 30, _no(t, 20, _no(t, 10), _no(t, 25)), _no(t, 40))
    r = t.rotacionar_direita(y)
    assert r.chave == 20 and r.direita.chave == 30 and r.direita.esquerda.chave == 25, "T2 não trocou de pai"
    assert (r.altura, r.direita.altura) == (2, 1), "atualize primeiro a altura do nó que desceu"

    # espelho: 10( -, 20( -, 30 ) ) -> 20(10, 30)
    x = _no(t, 10, None, _no(t, 20, None, _no(t, 30)))
    r = t.rotacionar_esquerda(x)
    assert (r.chave, r.esquerda.chave, r.direita.chave, r.altura) == (20, 10, 30, 1), "rotação à esquerda errada"
    assert t.rotacoes == 3

    # balancear: os 4 casos da tabela
    t2 = AVLTree()
    casos = {
        "LL": _no(t2, 30, _no(t2, 20, _no(t2, 10))),
        "LR": _no(t2, 30, _no(t2, 10, None, _no(t2, 20))),
        "RR": _no(t2, 10, None, _no(t2, 20, None, _no(t2, 30))),
        "RL": _no(t2, 10, None, _no(t2, 30, _no(t2, 20))),
    }
    gastos = {"LL": 1, "LR": 2, "RR": 1, "RL": 2}
    for nome, raiz in casos.items():
        antes = t2.rotacoes
        r = t2.balancear(raiz)
        assert (r.chave, r.esquerda.chave, r.direita.chave) == (20, 10, 30), f"caso {nome} errado"
        assert t2.rotacoes - antes == gastos[nome], f"caso {nome}: esperava {gastos[nome]} rotação(ões)"
    n = _no(t2, 20, _no(t2, 10))
    assert t2.balancear(n) is n, "nó balanceado: balancear deve devolver o próprio nó"
    print("✅ Questão 1 — nó, altura, FB e rotações ok!")

testar_q1_rotacoes()

✅ Questão 1 — nó, altura, FB e rotações ok!


In [6]:
# ===== TESTES DA QUESTÃO 1 — inserção (não altere) =====
def testar_q1_insercao():
    # sequência feita no quadro: 10, 20, 30, 40, 50, 25 (casos RR, RR e RL)
    t = AVLTree()
    for c in [10, 20, 30, 40, 50, 25]:
        t.adicionar(c, f"v{c}")
        verificar_avl(t)
    r = t.raiz
    assert (r.chave, r.esquerda.chave, r.direita.chave) == (30, 20, 40), "raiz/filhos errados"
    assert (r.esquerda.esquerda.chave, r.esquerda.direita.chave) == (10, 25)
    assert r.direita.esquerda is None and r.direita.direita.chave == 50
    assert t.rotacoes == 4, f"esperava 4 rotações (RR + RR + RL), houve {t.rotacoes}"
    assert len(t) == 6 and t.altura() == 2

    # os métodos herdados da ABB continuam funcionando
    assert coletar(t.em_ordem) == [10, 20, 25, 30, 40, 50]
    assert coletar(t.pre_ordem) == [30, 20, 10, 25, 40, 50]
    assert t.buscar(25) == "v25" and t.comparacoes == 3
    assert t.minimo() == 10 and t.maximo() == 50 and t.profundidade(50) == 2

    # chave repetida: atualiza o valor, sem mudar tamanho nem rotacionar
    t.adicionar(25, "novo")
    assert t.buscar(25) == "novo" and len(t) == 6 and t.rotacoes == 4

    # espelho feito pela turma: 50, 40, 30, 10, 20 (casos LL e LR)
    t = AVLTree()
    for c in [50, 40, 30, 10, 20]:
        t.adicionar(c)
    assert coletar(t.pre_ordem) == [40, 20, 10, 30, 50], "espelho (LL e LR) errado"
    assert t.rotacoes == 3, f"esperava 3 rotações (LL + LR), houve {t.rotacoes}"
    print("✅ Questão 1 — inserção ok!")

testar_q1_insercao()

✅ Questão 1 — inserção ok!


In [7]:
# ===== TESTES DA QUESTÃO 1 — remoção (não altere) =====
def testar_q1_remocao():
    t = AVLTree()
    assert t.remover(1) is False

    # exemplo da aula: 20(10, 30(25, 40)), remover 10 -> filho com FB = 0, rotação simples
    for c in [20, 10, 30, 25, 40]:
        t.adicionar(c)
    antes = t.rotacoes
    assert t.remover(10) is True and len(t) == 5 - 1
    assert coletar(t.pre_ordem) == [30, 20, 25, 40], "caso do filho com FB = 0 errado"
    assert t.rotacoes - antes == 1 and t.altura() == 2
    assert t.remover(10) is False and len(t) == 4
    verificar_avl(t)

    # nó com dois filhos: usa o sucessor
    t = AVLTree()
    for c in [50, 30, 70, 20, 40, 60, 80]:
        t.adicionar(c)
    t.remover(50)
    assert t.raiz.chave == 60, "dois filhos: o sucessor (60) deve ocupar o lugar de 50"
    t.remover(30)
    assert t.raiz.esquerda.chave == 40
    verificar_avl(t)

    # remover até esvaziar
    for c in [20, 40, 60, 70, 80]:
        assert t.remover(c) is True
        verificar_avl(t)
    assert t.raiz is None and len(t) == 0 and t.altura() == -1
    print("✅ Questão 1 — remoção ok!")

testar_q1_remocao()

✅ Questão 1 — remoção ok!


In [8]:
# ===== TESTES DA QUESTÃO 1 — estresse (não altere) =====
import math, random

def testar_q1_estresse():
    n = 3000
    t = AVLTree()
    for c in range(n):                      # chaves EM ORDEM: o pior caso da ABB
        t.adicionar(c, c * 10)
    verificar_avl(t)
    limite = 1.44 * math.log2(n + 2) - 0.328
    assert t.altura() <= limite, f"altura {t.altura()} acima do limite teórico da AVL ({limite:.1f})"
    assert t.buscar(n - 1) == (n - 1) * 10 and t.comparacoes <= t.altura() + 1

    rng = random.Random(7)
    saem = rng.sample(range(n), n // 2)
    for i, c in enumerate(saem, 1):
        assert t.remover(c) is True
        if i % 250 == 0:
            verificar_avl(t)
    ficaram = sorted(set(range(n)) - set(saem))
    assert len(t) == len(ficaram) and coletar(t.em_ordem) == ficaram
    assert all(not t.contem(c) for c in saem[:100])
    print(f"✅ Questão 1 — estresse ok! {n} chaves em ordem -> altura {t.altura()} "
          f"(uma ABB comum teria altura {n - 1}).")

testar_q1_estresse()

✅ Questão 1 — estresse ok! 3000 chaves em ordem -> altura 11 (uma ABB comum teria altura 2999).


### Por que `adicionar` e `remover` precisam ser sobrescritos?

1. **O que eles criam:** o `adicionar` da ABB cria objetos `No`, que não têm o campo `altura`. Na AVL todo nó precisa ser `AVLNode`.
2. **O que eles não fazem:** os métodos da ABB não atualizam alturas nem rebalanceiam o caminho até a raiz. Sem isso, a árvore pode degenerar e as alturas guardadas ficam erradas.

Os métodos que **só leem** a árvore (`buscar`, `contem`, `minimo`, `maximo`, `profundidade` e os percursos) funcionam sem alteração, porque uma AVL continua sendo uma ABB.

**Complexidade:** `altura`, `atualizar_altura`, `fator_balanceamento`, as rotações e `balancear` são O(1). `adicionar` e `remover` descem um caminho de tamanho O(log n) e chamam `balancear` (O(1)) em cada nó na volta, ou seja, **O(log n)**.

#### Verificação extra (minha): invariantes da AVL após inserções e remoções aleatórias

In [9]:
# ===== TESTE EXTRA (meu): sequências aleatórias de inserção/remoção =====
def testar_invariantes_aleatorio(rodadas=20, operacoes=600, semente=123):
    rng = random.Random(semente)
    for r in range(rodadas):
        t = AVLTree()
        referencia = {}                  # só para conferir o resultado (não resolve nada)
        for _ in range(operacoes):
            c = rng.randint(0, 300)
            if rng.random() < 0.6:
                t.adicionar(c, c * 2)
                referencia[c] = c * 2
            else:
                assert t.remover(c) == (c in referencia)
                referencia.pop(c, None)
        # verificar_avl confere ordem de ABB, alturas corretas e |FB| <= 1
        verificar_avl(t)
        # conferência explícita do FB em todos os nós
        fbs = []
        t.em_ordem(lambda no: fbs.append(t.fator_balanceamento(no)))
        assert all(-1 <= fb <= 1 for fb in fbs)
        assert coletar(t.em_ordem) == sorted(referencia)
        assert all(t.buscar(c) == v for c, v in referencia.items())
        n = len(t)
        assert n == len(referencia)
        assert n == 0 or t.altura() <= 1.44 * math.log2(n + 2) - 0.328
    print(f"✅ Invariantes AVL ok em {rodadas} rodadas de {operacoes} operações aleatórias!")

testar_invariantes_aleatorio()

✅ Invariantes AVL ok em 20 rodadas de 600 operações aleatórias!


---
## Questão 2 — Pedidos numerados em sequência

Uma loja virtual numera os pedidos **em sequência** (100001, 100002, 100003, …). O atendimento precisa localizar um pedido pelo número enquanto o cliente está no telefone. A equipe guardava os pedidos na ABB da prática anterior e reclama que, quanto mais a loja vende, mais lenta a consulta fica.

**(a)** Carregue os pedidos **na ordem em que chegaram** (a lista `pedidos`) em duas estruturas: `abb` (uma `ArvoreBinariaBusca`) e `avl` (uma `AVLTree`). Chave = número do pedido; valor = nome do cliente.

**(b)** `custo_medio_busca(arvore, chaves)`: busca cada chave da lista e retorna a **média** de nós visitados (o atributo `comparacoes`, que a AVL herdou da ABB).

**(c)** `altura_iterativa(arvore)`: retorna a altura de **qualquer** árvore binária (ABB ou AVL) **sem recursão** e **sem usar o campo `altura`**. Árvore vazia tem altura −1. Dica: um percurso em largura com uma fila de pares `(nó, nível)`.

**(d)** Execute a célula de comparação e responda à pergunta discursiva.

In [10]:
# ===== DADOS (não altere) =====
import random

_rng = random.Random(2026)
_CLIENTES = ["Ana", "Bruno", "Carla", "Diego", "Elisa", "Felipe", "Gabriela", "Heitor",
             "Isabela", "João", "Larissa", "Marcos", "Natália", "Otávio", "Paula", "Rafael"]

pedidos = [(100001 + i, _rng.choice(_CLIENTES)) for i in range(3000)]   # em ordem de chegada
consultas = _rng.sample([p for p, _ in pedidos], 300)

print(len(pedidos), "pedidos | primeiros:", pedidos[:3], "| últimos:", pedidos[-2:])

3000 pedidos | primeiros: [(100001, 'Diego'), (100002, 'Larissa'), (100003, 'Diego')] | últimos: [(102999, 'Carla'), (103000, 'Paula')]


In [11]:
# (a)
abb = ArvoreBinariaBusca()
avl = AVLTree()
# Carrega os pedidos na ordem de chegada (números crescentes) nas duas árvores
for numero, cliente in pedidos:
    abb.adicionar(numero, cliente)
    avl.adicionar(numero, cliente)


# (b)
def custo_medio_busca(arvore, chaves):
    # Soma os nós visitados (comparacoes) de cada busca e divide pela quantidade
    if not chaves:
        return 0.0
    total = 0
    for chave in chaves:
        arvore.buscar(chave)
        total += arvore.comparacoes
    return total / len(chaves)


# (c)
def altura_iterativa(arvore):
    # Percurso em largura com pares (nó, nível): a altura é o maior nível visto.
    # Não usa recursão (a ABB degenerada tem 3000 níveis) nem o campo `altura`.
    if arvore.raiz is None:
        return -1
    maior = 0
    fila = deque([(arvore.raiz, 0)])
    while fila:
        no, nivel = fila.popleft()
        if nivel > maior:
            maior = nivel
        if no.esquerda is not None:
            fila.append((no.esquerda, nivel + 1))
        if no.direita is not None:
            fila.append((no.direita, nivel + 1))
    return maior

In [12]:
# ===== TESTES DA QUESTÃO 2 (não altere) =====
assert len(abb) == 3000 and len(avl) == 3000, "as duas árvores deveriam ter 3000 pedidos"
verificar_avl(avl)
assert avl.buscar(pedidos[1234][0]) == pedidos[1234][1]

assert altura_iterativa(ArvoreBinariaBusca()) == -1
_t = AVLTree(); _t.adicionar(1)
assert altura_iterativa(_t) == 0
assert altura_iterativa(abb) == 2999, "pedidos em ordem: a ABB deveria virar uma fila"
assert altura_iterativa(avl) == avl.altura(), "altura_iterativa(avl) deveria bater com avl.altura()"

_esperado_abb = sum(c - 100000 for c in consultas) / len(consultas)
assert abs(custo_medio_busca(abb, consultas) - _esperado_abb) < 1e-9, "custo médio da ABB incorreto"
assert custo_medio_busca(avl, consultas) <= avl.altura() + 1
print("✅ Questão 2 (a), (b) e (c) ok!")

✅ Questão 2 (a), (b) e (c) ok!


In [13]:
# ===== COMPARAÇÃO (não altere) =====
print(f"{'':28}{'ABB':>10}{'AVL':>10}")
print(f"{'altura':28}{altura_iterativa(abb):>10}{avl.altura():>10}")
print(f"{'nós visitados por busca':28}{custo_medio_busca(abb, consultas):>10.1f}{custo_medio_busca(avl, consultas):>10.1f}")
print(f"{'rotações durante a carga':28}{'—':>10}{avl.rotacoes:>10}")
print(f"\nPiso teórico (árvore perfeitamente cheia): ⌊log2(3000)⌋ = {math.floor(math.log2(3000))}")

                                   ABB       AVL
altura                            2999        11
nós visitados por busca         1521.0      10.6
rotações durante a carga             —      2988

Piso teórico (árvore perfeitamente cheia): ⌊log2(3000)⌋ = 11


**(d)** Com base nos números da célula acima:

1. Explique por que a ABB ficou tão alta e por que isso torna cada consulta **O(n)**.
2. A AVL recebeu **as mesmas chaves, na mesma ordem**. Compare a altura dela com o piso teórico ⌊log₂ n⌋ e explique o que impediu a degeneração.
3. Nada é de graça: cite **duas** coisas que a AVL paga a mais que a ABB para manter esse desempenho.

✍️ **Sua resposta:**

1. **Por que a ABB ficou tão alta.** Os pedidos chegam com números crescentes (100001, 100002, …), então cada chave nova é maior que todas as anteriores e vai sempre para a **direita do último nó**. A ABB vira uma lista encadeada com **altura 2999 = n − 1**. Para achar o pedido de posição *i*, a busca passa por *i* nós. Por isso a média ficou em **1521 nós por busca**, cerca de n/2. O custo cresce linearmente com o número de pedidos, ou seja, cada consulta é **O(n)**. Isso explica a reclamação de que "quanto mais a loja vende, mais lento fica".

2. **A AVL com as mesmas chaves.** A AVL ficou com **altura 11**, exatamente o piso teórico ⌊log₂ 3000⌋ = 11, e cada busca visitou em média **10,6 nós**, contra 1521 na ABB. Quem impediu a degeneração foi o **rebalanceamento**. A cada inserção, as alturas são atualizadas no caminho de volta até a raiz. Quando algum nó fica com FB = −2 (aqui sempre o caso RR, porque as chaves só crescem), uma rotação à esquerda corrige a subárvore. Assim a altura fica garantida em O(log n), e a busca também.

3. **O que a AVL paga a mais:**
   - **Tempo extra nas atualizações:** cada inserção ou remoção recalcula alturas no caminho e pode fazer rotações. Só na carga foram **2988 rotações**, quase uma por pedido. A complexidade continua O(log n), mas com uma constante maior que a da ABB.
   - **Memória extra:** cada nó guarda o campo `altura`.
   - Além disso, o código fica mais complexo (rotações e os quatro casos), e a versão recursiva usa pilha de chamadas, que é segura só porque a altura é O(log n).

---
## Questão 3 — Livro de ofertas de uma corretora

Uma corretora mantém o **livro de ofertas de compra** de uma ação: para cada **preço** (em centavos, para evitar erros de arredondamento), a **quantidade total** de ações que os compradores aceitam pagar àquele preço. Ofertas chegam o tempo todo, e quando aparece um vendedor ele é atendido **primeiro pelos maiores preços**. Muitas inserções **e** muitas remoções: é o cenário em que o rebalanceamento da AVL trabalha nos dois sentidos.

Use uma `AVLTree` com **chave = preço** e **valor = quantidade**.

**(a)** `registrar_oferta(livro, preco, quantidade)`: se o preço ainda não está no livro, cria a entrada; se já está, **soma** a quantidade à existente.

**(b)** `melhor_oferta(livro)`: retorna a tupla `(preco, quantidade)` do **maior** preço, ou `None` se o livro estiver vazio.

**(c)** `vender(livro, quantidade)`: um vendedor quer vender `quantidade` ações. Atenda pelos maiores preços primeiro: um preço totalmente consumido sai do livro (`remover`); um preço consumido em parte fica com o que sobrou. Pare quando a venda for completada ou o livro acabar. Retorne a lista de negócios `[(preco, quantidade_negociada), ...]` na ordem em que aconteceram.

**(d)** `volume_entre(livro, pmin, pmax)`: retorna a tupla `(volume, visitados)`, em que `volume` é a soma das quantidades com `pmin ≤ preço ≤ pmax` e `visitados` é o número de nós que a sua função visitou. **Não percorra a árvore inteira:** a partir da raiz, só desça para a esquerda se ali puder haver preços ≥ `pmin`, e só desça para a direita se ali puder haver preços ≤ `pmax`. (Recursão é segura aqui. Por quê?)

In [14]:
# ===== DADOS (não altere) =====
_rng = random.Random(42)
ofertas = [(_rng.randint(9000, 11000), _rng.choice(range(100, 1001, 100))) for _ in range(5000)]
vendas = [_rng.choice([300, 800, 1500, 4000, 12000]) for _ in range(200)]
print(len(ofertas), "ofertas | exemplos:", ofertas[:4])

5000 ofertas | exemplos: [(10309, 200), (9051, 500), (9501, 400), (9285, 200)]


In [15]:
# (a)
def registrar_oferta(livro, preco, quantidade):
    # buscar() devolve None se o preço não existe; senão soma à quantidade atual
    atual = livro.buscar(preco)
    if atual is None:
        livro.adicionar(preco, quantidade)
    else:
        livro.adicionar(preco, atual + quantidade)   # chave repetida: só atualiza o valor


# (b)
def melhor_oferta(livro):
    # O maior preço é o nó mais à direita: O(log n) na AVL
    if livro.raiz is None:
        return None
    no = livro.raiz
    while no.direita is not None:
        no = no.direita
    return (no.chave, no.valor)


# (c)
def vender(livro, quantidade):
    negocios = []
    restante = quantidade
    while restante > 0:
        melhor = melhor_oferta(livro)
        if melhor is None:              # livro acabou
            break
        preco, disponivel = melhor
        negociado = min(restante, disponivel)
        negocios.append((preco, negociado))
        restante -= negociado
        if negociado == disponivel:
            livro.remover(preco)        # preço totalmente consumido sai do livro
        else:
            livro.adicionar(preco, disponivel - negociado)   # sobra no mesmo preço
    return negocios


# (d)
def volume_entre(livro, pmin, pmax):
    # Busca por intervalo com poda: recursão segura porque a profundidade
    # é a altura da AVL, O(log n) (~1,44 log2 n no pior caso).
    visitados = 0

    def _volume(no):
        nonlocal visitados
        if no is None:
            return 0
        visitados += 1
        total = 0
        if no.chave > pmin:             # só à esquerda pode haver preços >= pmin
            total += _volume(no.esquerda)
        if pmin <= no.chave <= pmax:
            total += no.valor
        if no.chave < pmax:             # só à direita pode haver preços <= pmax
            total += _volume(no.direita)
        return total

    volume = _volume(livro.raiz)
    return (volume, visitados)

In [16]:
# ===== TESTES DA QUESTÃO 3 (não altere) =====
from collections import Counter

livro = AVLTree()
for p, q in ofertas:
    registrar_oferta(livro, p, q)

_ref = Counter()
for p, q in ofertas:
    _ref[p] += q

assert len(livro) == len(_ref), "cada preço deve aparecer uma única vez no livro"
assert livro.buscar(ofertas[0][0]) == _ref[ofertas[0][0]], "as quantidades de um mesmo preço devem ser somadas"
verificar_avl(livro)
_pmax = max(_ref)
assert melhor_oferta(livro) == (_pmax, _ref[_pmax])
assert melhor_oferta(AVLTree()) is None

# (d) antes das vendas
for a, b in [(9000, 11000), (9950, 10050), (10500, 10500), (8000, 8999), (10990, 12000)]:
    vol, vis = volume_entre(livro, a, b)
    esperado = sum(q for p, q in _ref.items() if a <= p <= b)
    k = sum(1 for p in _ref if a <= p <= b)
    assert vol == esperado, f"volume_entre({a}, {b}) = {vol}, esperado {esperado}"
    assert vis <= k + 2 * (livro.altura() + 1), (
        f"volume_entre({a}, {b}) visitou {vis} nós: está descendo onde não precisa?")

# (c) vendas, conferidas contra uma simulação de referência
_precos = sorted(_ref)
for q in vendas:
    negocios = vender(livro, q)
    total_ref, esperado = q, []
    while total_ref > 0 and _precos:
        p = _precos[-1]
        n = min(total_ref, _ref[p])
        esperado.append((p, n))
        total_ref -= n
        _ref[p] -= n
        if _ref[p] == 0:
            del _ref[p]
            _precos.pop()
    assert negocios == esperado, f"venda de {q} ações: negócios diferentes do esperado"
verificar_avl(livro)
assert len(livro) == len(_ref) and melhor_oferta(livro) == ((_precos[-1], _ref[_precos[-1]]) if _precos else None)
assert vender(AVLTree(), 100) == []

print(f"✅ Questão 3 ok! Restaram {len(livro)} preços no livro; altura da AVL = {livro.altura()}; "
      f"rotações desde a criação = {livro.rotacoes}.")

✅ Questão 3 ok! Restaram 1397 preços no livro; altura da AVL = 12; rotações desde a criação = 1640.


**Observações da Questão 3**

- **Por que a recursão é segura em `volume_entre`?** A profundidade da recursão é no máximo a altura da árvore, e numa AVL essa altura é O(log n) (≤ 1,44·log₂ n). Com ~2000 preços isso dá uns 15 níveis, muito longe do limite de recursão do Python. Numa ABB degenerada a recursão poderia ter milhares de níveis.
- **Complexidade:** `registrar_oferta` e `melhor_oferta` são O(log n). `vender` faz O(log n) por preço consumido, ou seja, O(t · log n) para t negócios. `volume_entre` visita O(k + log n) nós (k = preços no intervalo), porque a poda só desce por subárvores que podem ter preços dentro de [pmin, pmax].

---
## Questão 4 — Ranking ao vivo de um jogo

Um jogo online mostra um **ranking ao vivo**. A cada partida, a pontuação de um jogador muda e o ranking precisa refletir isso na hora. O sistema precisa de duas consultas rápidas: "**quantos pontos tem o jogador X?**" e "**quem está no topo?**". Uma única árvore não responde bem às duas, então o `Placar` usa **duas AVLs**:

- `self.pontos`: chave = nome do jogador, valor = pontos (o "índice" por jogador);
- `self.ranking`: chave = a tupla `(-pontos, jogador)`, valor = `None`.

Em Python, tuplas são comparadas elemento a elemento: `(-900, "ana") < (-500, "bia")` e `(-500, "ana") < (-500, "bia")`. Assim, o percurso **em-ordem** de `ranking` lista do **maior** para o **menor** número de pontos, desempatando pelo nome.

**(a)** `cadastrar(jogador, pontos)`: inclui um jogador novo nas **duas** árvores.

**(b)** `registrar_partida(jogador, ganho)`: soma `ganho` aos pontos do jogador, mantendo as **duas** árvores consistentes, e retorna a nova pontuação. Pense: o que acontece com a chave antiga do jogador em `ranking`?

**(c)** `top(k)`: retorna a lista `[(jogador, pontos), ...]` dos `k` primeiros do ranking, usando um percurso com uma função `processar(no)`.

**(d)** Execute os dois experimentos e responda à pergunta discursiva.

In [17]:
# ===== DADOS (não altere) =====
_rng = random.Random(99)
jogadores = [(f"jogador{i:03d}", _rng.randint(0, 2000)) for i in range(1, 501)]
partidas = [(_rng.choice(jogadores)[0], _rng.randint(0, 150)) for _ in range(6000)]
print(len(jogadores), "jogadores |", len(partidas), "partidas | exemplo:", partidas[:3])

500 jogadores | 6000 partidas | exemplo: [('jogador083', 87), ('jogador396', 13), ('jogador458', 87)]


In [18]:
class _TopCompleto(Exception):
    """Usada para interromper o percurso em-ordem assim que já temos k jogadores."""


class Placar:
    def __init__(self):
        self.pontos = AVLTree()    # jogador -> pontos
        self.ranking = AVLTree()   # (-pontos, jogador) -> None

    # (a)
    def cadastrar(self, jogador, pontos):
        # Índice por jogador e ranking (pontos negativos: em-ordem = maior primeiro)
        self.pontos.adicionar(jogador, pontos)
        self.ranking.adicionar((-pontos, jogador))

    # (b)
    def registrar_partida(self, jogador, ganho):
        antigos = self.pontos.buscar(jogador)
        novos = antigos + ganho
        # A chave do ranking depende dos pontos: a antiga precisa SAIR,
        # senão o jogador apareceria duas vezes. Remove e insere a nova.
        self.ranking.remover((-antigos, jogador))
        self.ranking.adicionar((-novos, jogador))
        self.pontos.adicionar(jogador, novos)        # chave existente: atualiza o valor
        return novos

    # (c)
    def top(self, k):
        resultado = []
        if k <= 0:
            return resultado

        def processar(no):
            neg_pontos, jogador = no.chave
            resultado.append((jogador, -neg_pontos))
            if len(resultado) >= k:
                raise _TopCompleto      # para o percurso cedo: O(log n + k)

        try:
            self.ranking.em_ordem(processar)
        except _TopCompleto:
            pass
        return resultado

In [19]:
# ===== TESTES DA QUESTÃO 4 (não altere) =====
placar = Placar()
for j, p in jogadores:
    placar.cadastrar(j, p)
assert len(placar.pontos) == 500 and len(placar.ranking) == 500

_ref = {j: p for j, p in jogadores}
_top_ref = lambda k: sorted(_ref.items(), key=lambda jp: (-jp[1], jp[0]))[:k]
assert placar.top(5) == _top_ref(5), "top(5) logo após o cadastro está errado"

for j, g in partidas:
    _ref[j] += g
    assert placar.registrar_partida(j, g) == _ref[j]

assert len(placar.ranking) == 500, "o ranking deve continuar com 500 entradas: a chave antiga saiu?"
assert placar.top(10) == _top_ref(10), "top(10) após as partidas está errado"
assert placar.top(1000) == _top_ref(500)
assert placar.pontos.buscar("jogador042") == _ref["jogador042"]
verificar_avl(placar.pontos)
verificar_avl(placar.ranking)
print("✅ Questão 4 (a), (b) e (c) ok!  Top 3:", placar.top(3))

✅ Questão 4 (a), (b) e (c) ok!  Top 3: [('jogador114', 3372), ('jogador273', 3316), ('jogador314', 3234)]


In [20]:
# ===== EXPERIMENTO 1: quanto custa manter o ranking balanceado? (não altere) =====
_rng = random.Random(5)
r = placar.ranking
ins, rem, max_rem, com_rotacao_rem = [], [], 0, 0
for _ in range(5000):
    j = _rng.choice(jogadores)[0]
    p = placar.pontos.buscar(j)
    novo = p + _rng.randint(0, 150)
    a = r.rotacoes; r.remover((-p, j)); d_rem = r.rotacoes - a
    a = r.rotacoes; r.adicionar((-novo, j)); d_ins = r.rotacoes - a
    placar.pontos.adicionar(j, novo)
    rem.append(d_rem); ins.append(d_ins)
    max_rem = max(max_rem, d_rem)
verificar_avl(r)
print(f"Ranking com {len(r)} jogadores, altura {r.altura()}")
print(f"rotações por inserção: média {sum(ins)/len(ins):.2f}, máximo {max(ins)}")
print(f"rotações por remoção:  média {sum(rem)/len(rem):.2f}, máximo {max_rem}")

Ranking com 500 jogadores, altura 10
rotações por inserção: média 0.64, máximo 2
rotações por remoção:  média 0.34, máximo 5


In [21]:
# ===== EXPERIMENTO 2: uma remoção que rebalanceia em dois níveis (não altere) =====
fib = AVLTree()
for c in [8, 5, 11, 3, 7, 10, 12, 2, 4, 6, 9, 1]:
    fib.adicionar(c)
print(f"ANTES (rotações na construção: {fib.rotacoes}, altura {fib.altura()}):")
mostrar(fib)
antes = fib.rotacoes
fib.remover(12)
verificar_avl(fib)
print(f"\nDEPOIS de remover a folha 12 (rotações: {fib.rotacoes - antes}, altura {fib.altura()}):")
mostrar(fib)

ANTES (rotações na construção: 0, altura 4):
                12  [h=0, FB=0]
        11  [h=2, FB=1]
                10  [h=1, FB=1]
                        9  [h=0, FB=0]
8  [h=4, FB=1]
                7  [h=1, FB=1]
                        6  [h=0, FB=0]
        5  [h=3, FB=1]
                        4  [h=0, FB=0]
                3  [h=2, FB=1]
                        2  [h=1, FB=1]
                                1  [h=0, FB=0]

DEPOIS de remover a folha 12 (rotações: 2, altura 3):
                        11  [h=0, FB=0]
                10  [h=1, FB=0]
                        9  [h=0, FB=0]
        8  [h=2, FB=0]
                7  [h=1, FB=1]
                        6  [h=0, FB=0]
5  [h=3, FB=0]
                4  [h=0, FB=0]
        3  [h=2, FB=1]
                2  [h=1, FB=1]
                        1  [h=0, FB=0]


**(d)** Com base nos dois experimentos:

1. No Experimento 2, a remoção de **uma folha** causou rotações em **dois nós diferentes**. Identifique esses nós e explique por que o primeiro rebalanceamento não resolveu tudo. Por que isso **não** pode acontecer na inserção?
2. Uma colega sugere trocar o `ranking` por uma **lista ordenada**, porque "o top-k fica trivial: é só pegar os k primeiros". Compare o custo de **uma** chamada a `registrar_partida` nas duas abordagens e diga qual você recomendaria para um jogo com milhões de jogadores.

✍️ **Sua resposta:**

1. **Experimento 2: rotações em dois nós.** Antes da remoção, o nó **11** tinha FB = +1 (10 com altura 1 à esquerda, 12 com altura 0 à direita), e a raiz **8** tinha FB = +1 (5 com altura 3, 11 com altura 2).
   - Quando a folha 12 sai, o **nó 11** fica com FB = +2, e o filho 10 tem FB = +1. Isso é o caso LL, resolvido com uma **rotação simples à direita em 11**. A subárvore passa a ter raiz 10 com filhos 9 e 11.
   - Esse primeiro rebalanceamento **não resolveu tudo**, porque a subárvore direita da raiz ficou **mais baixa**. Ela tinha altura 2 e passou a ter altura 1, já que a remoção diminui a altura e a rotação não devolve a altura antiga. Com isso, a **raiz 8** ficou com FB = 3 − 1 = +2, e o filho 5 tem FB = +1. É de novo o caso LL, resolvido com uma **rotação simples à direita em 8**, e 5 vira a raiz. Foram 2 rotações no total, e a altura caiu de 4 para 3.
   - **Por que isso não acontece na inserção?** Na inserção, a subárvore desbalanceada tinha **crescido** 1 de altura. A rotação (simples ou dupla) a devolve **exatamente à altura que tinha antes da inserção**, então os ancestrais não percebem mudança nenhuma. Por isso basta **um** rebalanceamento por inserção, com no máximo 2 rotações. Na remoção, a rotação pode deixar a subárvore 1 nível mais baixa, e esse "encolhimento" se propaga para cima. Pode haver rotações em até O(log n) nós. O Experimento 1 confirma isso: na inserção o máximo foi **2** rotações, e na remoção chegou a **5**.

2. **Lista ordenada × AVL em `registrar_partida`.**
   - **Lista ordenada:** a busca binária acha a posição antiga em O(log n). Mas **remover** a entrada antiga e **inserir** a nova na posição certa desloca elementos do array, o que custa **O(n)** no pior caso. Com 1 milhão de jogadores, isso pode significar mover centenas de milhares de elementos a cada partida. O top-k realmente fica trivial, com custo O(k).
   - **AVL (`Placar`):** uma busca em `pontos`, uma remoção e uma inserção em `ranking` e uma atualização em `pontos`, todas **O(log n)**. Com 1 milhão de jogadores são cerca de 20 a 30 nós por operação. O `top(k)` também é eficiente: O(log n + k), porque o percurso em-ordem para assim que junta k jogadores.
   - **Recomendação:** para um jogo com milhões de jogadores e atualizações a cada partida, a **AVL**. As atualizações são muito mais frequentes que a consulta do top, e o top-k continua barato. A lista ordenada só compensaria se o ranking quase nunca mudasse.

---
### ✅ Antes de enviar

- [ ] *Kernel → Restart & Run All* executou sem erros
- [ ] Todas as células de teste mostram ✅
- [ ] Todas as respostas ✍️ foram preenchidas
- [ ] Nome e matrícula no topo do notebook